In [2]:
# ============================================================================
# 导入必要的库
# ============================================================================
import os
import mne
import numpy as np
import pandas as pd
from scipy import interpolate, signal
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# ============================================================================
# 配置参数
# ============================================================================
# 数据根目录
ROOT_DIR = "/root/autodl-tmp/ADFTD Sub88"

# 重采样参数
ORIGINAL_SAMPLING_RATE = 500  # 原始采样率 (Hz)
TARGET_SAMPLING_RATE = 128    # 目标采样率 (Hz)
RESAMPLING_KIND = 'linear'    # 插值方法: 'linear', 'cubic', 'nearest'

# 窗口分割参数
WINDOW_TIME = 1.0              # 窗口大小（秒）
OVERLAP_RATIO = 0.0            # 窗口重叠比例 (0.0-0.5)，0.0表示无重叠
# 根据目标采样率计算窗口大小（时间点数）
WINDOW_SIZE = int(WINDOW_TIME * TARGET_SAMPLING_RATE)
# 是否进行划窗；False 时不切分，整段重采样序列作为一个样本
ENABLE_WINDOWING = True
# 是否进行窗口质量检测；False 时保留全部窗口（未划窗时则对整段检测）
ENABLE_WINDOW_QC = True

# 是否启用数据增强（Random Cropping 双路）；True 时保存形状为 (2, N, WINDOW_SIZE, n_channels)
# 其中 array[0] 为原始切片，array[1] 为增强切片，同一 N 下标严格对齐
# 关闭划窗时 Random Crop 不生效，两路保存同一整段
USE_AUGMENTATION = True
AUGMENTATION_RANDOM_SEED = 42

# 输出路径（默认划窗 + QC 时目录名与原来一致；关闭时追加后缀，避免覆盖已有结果）
output_base_dir = r"/root/autodl-tmp/ADMamba/dataset"
output_tags = []
if not ENABLE_WINDOWING:
    output_tags.append("NoWin")
if not ENABLE_WINDOW_QC:
    output_tags.append("NoQC")
if USE_AUGMENTATION:
    output_tags.append("Aug")
tag_str = ("_" + "_".join(output_tags)) if output_tags else ""
output_dir_name = f"ADFTD3_{WINDOW_TIME}s_{OVERLAP_RATIO}{tag_str}"
OUTPUT_DIR = os.path.join(output_base_dir, output_dir_name)
OUTPUT_LABEL_DIR = os.path.join(OUTPUT_DIR, 'Label')
OUTPUT_FEATURE_DIR = os.path.join(OUTPUT_DIR, 'Feature')

# 标签映射
LABEL_MAP = {'A': 2, 'F': 1, 'C': 0}  # A:2, F:1, C:0

print("=" * 60)
print("配置参数:")
print(f"  数据根目录: {ROOT_DIR}")
print(f"  原始采样率: {ORIGINAL_SAMPLING_RATE} Hz")
print(f"  目标采样率: {TARGET_SAMPLING_RATE} Hz")
print(f"  窗口时间: {WINDOW_TIME} s")
print(f"  窗口大小: {WINDOW_SIZE} 时间点数")
print(f"  重叠比例: {OVERLAP_RATIO * 100:.1f}%")
print(f"  划窗处理: {'开启' if ENABLE_WINDOWING else '关闭'}")
print(f"  窗口质量检测: {'开启' if ENABLE_WINDOW_QC else '关闭'}")
if USE_AUGMENTATION and not ENABLE_WINDOWING:
    print("  注意: 已关闭划窗，Random Crop 增强不生效，两路保存同一整段")
print(f"  输出基础目录: {output_base_dir}")
print(f"  输出目录名称: {output_dir_name}")
print(f"  输出目录: {OUTPUT_DIR}")
print("=" * 60)

配置参数:
  数据根目录: /root/autodl-tmp/ADFTD Sub88
  原始采样率: 500 Hz
  目标采样率: 128 Hz
  窗口时间: 1.0 s
  窗口大小: 128 时间点数
  重叠比例: 0.0%
  划窗处理: 关闭
  窗口质量检测: 关闭
  输出基础目录: /root/autodl-tmp/ADMamba/dataset
  输出目录名称: ADFTD3_1.0s_0.0_NoWin_NoQC
  输出目录: /root/autodl-tmp/ADMamba/dataset/ADFTD3_1.0s_0.0_NoWin_NoQC


In [10]:
# ============================================================================
# 数据路径设置
# ============================================================================
participants_path = os.path.join(ROOT_DIR, 'participants.tsv')
derivatives_root = os.path.join(ROOT_DIR, 'derivatives/')

# 创建输出目录
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)
    print(f"创建输出目录: {OUTPUT_DIR}")

if not os.path.exists(OUTPUT_LABEL_DIR):
    os.makedirs(OUTPUT_LABEL_DIR)
    print(f"创建标签输出目录: {OUTPUT_LABEL_DIR}")

if not os.path.exists(OUTPUT_FEATURE_DIR):
    os.makedirs(OUTPUT_FEATURE_DIR)
    print(f"创建特征输出目录: {OUTPUT_FEATURE_DIR}")

print(f"参与者文件路径: {participants_path}")
print(f"衍生数据根目录: {derivatives_root}")

创建输出目录: /root/autodl-tmp/ADMamba/dataset/ADFTD3_1.0s_0.0_NoWin_NoQC
创建标签输出目录: /root/autodl-tmp/ADMamba/dataset/ADFTD3_1.0s_0.0_NoWin_NoQC/Label
创建特征输出目录: /root/autodl-tmp/ADMamba/dataset/ADFTD3_1.0s_0.0_NoWin_NoQC/Feature
参与者文件路径: /root/autodl-tmp/ADFTD Sub88/participants.tsv
衍生数据根目录: /root/autodl-tmp/ADFTD Sub88/derivatives/


In [11]:
# ============================================================================
# 读取参与者信息并生成标签
# ============================================================================
participants = pd.read_csv(participants_path, sep='\t')
print(f"参与者数量: {participants.shape[0]}")
print("\n参与者信息预览:")
print(participants.head())

# 创建标签数组 [label, participant_id]
labels = np.empty(shape=(participants.shape[0], 2), dtype='int32')

for i, participant in enumerate(participants.values):
    pid = int(participant[0][-3:])  # 提取参与者ID
    label = LABEL_MAP[participant[3]]  # 获取标签
    labels[i, 0] = label
    labels[i, 1] = pid

# 保存标签
label_save_path = os.path.join(OUTPUT_LABEL_DIR, 'label.npy')
np.save(label_save_path, labels)
print(f"\n标签已保存至: {label_save_path}")
print(f"标签形状: {labels.shape}")
print(f"标签分布: {np.bincount(labels[:, 0])}")

参与者数量: 88

参与者信息预览:
  participant_id Gender  Age Group  MMSE
0        sub-001      F   57     A    16
1        sub-002      F   78     A    22
2        sub-003      M   70     A    14
3        sub-004      F   67     A    20
4        sub-005      M   70     A    22

标签已保存至: /root/autodl-tmp/ADMamba/dataset/ADFTD3_1.0s_0.0_NoWin_NoQC/Label/label.npy
标签形状: (88, 2)
标签分布: [29 23 36]


In [12]:
# ============================================================================
# 数据质量检查（可选）
# ============================================================================
# 检查bad channels、采样频率和数据形状
bad_channel_list, sampling_freq_list, data_shape_list = [], [], []

for sub in os.listdir(derivatives_root):
    if 'sub-' in sub:
        sub_path = os.path.join(derivatives_root, sub, 'eeg/')
        for file in os.listdir(sub_path):
            if '.set' in file:
                file_path = os.path.join(sub_path, file)
                raw = mne.io.read_raw_eeglab(file_path, preload=False)
                # 获取bad channels
                bad_channel = raw.info['bads']
                bad_channel_list.append(bad_channel)
                # 获取采样频率
                sampling_freq = raw.info['sfreq']
                sampling_freq_list.append(sampling_freq)
                # 获取EEG数据形状
                data = raw.get_data()
                data_shape = data.shape
                data_shape_list.append(data_shape)

print("数据质量检查结果:")
print(f"  Bad channels: {bad_channel_list}")
print(f"  采样频率: {sampling_freq_list}")
print(f"  数据形状: {data_shape_list}")

Reading /root/autodl-tmp/ADFTD Sub88/derivatives/sub-001/eeg/sub-001_task-eyesclosed_eeg.set
Reading /root/autodl-tmp/ADFTD Sub88/derivatives/sub-002/eeg/sub-002_task-eyesclosed_eeg.set
Reading /root/autodl-tmp/ADFTD Sub88/derivatives/sub-003/eeg/sub-003_task-eyesclosed_eeg.set
Reading /root/autodl-tmp/ADFTD Sub88/derivatives/sub-004/eeg/sub-004_task-eyesclosed_eeg.set
Reading /root/autodl-tmp/ADFTD Sub88/derivatives/sub-005/eeg/sub-005_task-eyesclosed_eeg.set
Reading /root/autodl-tmp/ADFTD Sub88/derivatives/sub-006/eeg/sub-006_task-eyesclosed_eeg.set
Reading /root/autodl-tmp/ADFTD Sub88/derivatives/sub-007/eeg/sub-007_task-eyesclosed_eeg.set
Reading /root/autodl-tmp/ADFTD Sub88/derivatives/sub-008/eeg/sub-008_task-eyesclosed_eeg.set
Reading /root/autodl-tmp/ADFTD Sub88/derivatives/sub-009/eeg/sub-009_task-eyesclosed_eeg.set
Reading /root/autodl-tmp/ADFTD Sub88/derivatives/sub-010/eeg/sub-010_task-eyesclosed_eeg.set
Reading /root/autodl-tmp/ADFTD Sub88/derivatives/sub-011/eeg/sub-011_t

In [13]:
# ============================================================================
# 预处理函数定义
# ============================================================================

def resampling(array, original_freq, target_freq, kind='linear'):
    """
    重采样函数
    
    参数:
        array: 输入信号数组
        original_freq: 原始采样率
        target_freq: 目标采样率
        kind: 插值方法 ('linear', 'cubic', 'nearest')
    
    返回:
        重采样后的数组
    """
    t = np.linspace(1, len(array), len(array))
    f = interpolate.interp1d(t, array, kind=kind)
    t_new = np.linspace(1, len(array), int(len(array) / original_freq * target_freq))
    new_array = f(t_new)
    return new_array


def window_qc(
    windows,
    sfreq,
    n_channels,
    bad_channel_ratio_thresh=0.2,  # >20% 通道坏 → 丢窗
    abs_amp_thresh=200.0            # µV，保护性绝对阈值
):
    """
    对窗口进行质量检测，识别异常窗口。

    参数:
    - windows (np.ndarray): 窗口数据，3D数组
    - sfreq (float): 采样率 (Hz)
    - n_channels (int): 通道数，用于判断和修正维度顺序
    - bad_channel_ratio_thresh (float): 坏通道比例阈值，超过此比例的窗口将被标记为异常。默认: 0.2
    - abs_amp_thresh (float): 绝对幅值阈值 (µV)。默认: 200.0

    返回:
    - qc_pass (np.ndarray): 布尔数组，形状为 (n_windows,)，True 表示通过 QC
    """
    # 验证输入是3D数组
    if windows.ndim != 3:
        raise ValueError(f"windows 必须是 3D 数组，当前形状: {windows.shape}")
    
    # 检查最后一个维度是否为通道数
    original_shape = windows.shape
    
    # 如果最后一个维度是通道数，交换后两个维度
    if windows.shape[-1] == n_channels:
        # 交换后两个维度：(..., dim1, dim2) -> (..., dim2, dim1)
        windows = np.transpose(windows, (0, 2, 1))
        print(f"  形状修正: {original_shape} -> {windows.shape} (交换后两个维度)")
    
    n_w, n_ch, n_t = windows.shape

    # ---------- A. 峰峰值 ----------
    ptp = np.ptp(windows, axis=-1)        # (n_w, n_ch)
    ptp_med = np.median(ptp, axis=0)
    ptp_mad = np.median(np.abs(ptp - ptp_med), axis=0) + 1e-6
    ptp_bad = ptp > (ptp_med + 6 * ptp_mad)

    # 绝对幅值保护
    absmax = np.max(np.abs(windows), axis=-1)
    abs_bad = absmax > abs_amp_thresh

    # ---------- B. 汇总坏道 ----------
    bad_channel_mask = ptp_bad | abs_bad
    bad_ratio = bad_channel_mask.sum(axis=1) / n_ch
    print(f"  坏道数量: {bad_channel_mask.sum(axis=1)}")

    qc_pass = bad_ratio <= bad_channel_ratio_thresh

    return qc_pass



def segment(df, window_size, overlap_ratio=0.0, sfreq=128, enable_qc=True, 
            bad_channel_ratio_thresh=0.2, abs_amp_thresh=200.0):
    """
    窗口分割函数（支持重叠和质量控制）
    
    参数:
        df: 输入DataFrame
        window_size: 窗口大小（时间点数）
        overlap_ratio: 重叠比例 (0.0-1.0)，0.0表示无重叠
        sfreq: 采样率 (Hz)，默认 128
        enable_qc: 是否启用质量控制，默认 True
        bad_channel_ratio_thresh: 坏通道比例阈值，默认 0.2
        abs_amp_thresh: 绝对幅值阈值 (µV)，默认 200.0
    
    返回:
        分割后的DataFrame列表
    """
    res = []
    data_length = df.shape[0]
    
    # 计算步长（stride）
    stride = int(window_size * (1 - overlap_ratio))
    
    # 从中间位置开始分割
    start = int(data_length / 2)
    
    # 向左扩展
    left_index = start - int(start / stride) * stride
    
    # 向右扩展
    right_index = start + int((data_length - start) / stride) * stride
    
    # 生成窗口
    for i in range(left_index, right_index, stride):
        if i + window_size <= data_length:
            res.append(df.iloc[i: i + window_size, :])
    
    # 如果启用了质量控制
    if enable_qc and len(res) > 0:
        # 将 DataFrame 列表转换为 numpy 数组 (n_windows, n_channels, n_times)
        windows_array = np.array([window.values.T for window in res])
        
        # 执行质量控制
        qc_pass = window_qc(
            windows_array, 
            sfreq,
            n_channels=df.shape[1],  # 通道数
            bad_channel_ratio_thresh=bad_channel_ratio_thresh,
            abs_amp_thresh=abs_amp_thresh
        )
        
        # 筛选通过质量控制的窗口
        res_filtered = [res[i] for i in range(len(res)) if qc_pass[i]]
        
        # 打印质量控制统计信息
        print(f"  质量控制: 总窗口数 {len(res)}, 通过 {qc_pass.sum()}, 丢弃 {(~qc_pass).sum()}")
        
        return res_filtered
    
    return res


def segment_random_crop_pair(
    df,
    window_size,
    overlap_ratio=0.0,
    sfreq=128,
    enable_qc=True,
    bad_channel_ratio_thresh=0.2,
    abs_amp_thresh=200.0,
    rng=None,
    random_seed=42,
):
    """
    窗口分割 + Random Cropping 数据增强：生成**成对**且**窗口一一对应**的两条切片序列。

    在确定可用长度时，按规则在序列末尾预留一段用于随机起点：
    - 若从起点 0、步长 stride 铺窗后尾部无剩余（完美铺满），则预留 1 个窗口长度；
    - 否则若剩余长度 < 半个窗口，则预留 2 个窗口长度；
    - 其余情况预留 1 个窗口长度。

    在 [0, L_work) 内取两个首窗起点 start_ori、start_aug，满足
    |start_aug - start_ori| ∈ [window_size/10, window_size/2]（闭区间取整后落在整数格点上）。
    两路使用相同 stride、相同窗口数 n_windows。

    QC：对两路分别做 window_qc 后，**按索引合并**——仅当同一下标两窗均通过时才保留，
    保证第 k 个窗口在两条序列中始终对应同一时间顺序（任一路该位失败则丢弃该对）。

    参数:
        df: 输入 DataFrame（行=时间，列=通道）
        window_size: 窗口长度（采样点）
        overlap_ratio: 重叠比例 [0,1)
        rng: 可选 ``np.random.Generator``；若为 None，则用 ``random_seed`` 创建（默认 42）
        random_seed: Random Cropping 随机种子，默认 42

    返回:
        (list[DataFrame], list[DataFrame], dict)
        第三项包含 start_ori, start_aug, gap, n_windows, reserved_tail 等调试信息。
    """
    if rng is None:
        rng = np.random.default_rng(random_seed)

    data_length = df.shape[0]
    W = int(window_size)
    stride = max(1, int(round(W * (1 - float(overlap_ratio)))))

    empty_meta = {
        "start_ori": None,
        "start_aug": None,
        "gap": None,
        "n_windows": 0,
        "reserved_tail": None,
    }
    if data_length < W:
        return [], [], empty_meta

    last_start = ((data_length - W) // stride) * stride
    tail = data_length - (last_start + W)

    if tail == 0:
        reserved_tail = W
    elif tail < (W / 2.0):
        reserved_tail = 2 * W
    else:
        reserved_tail = W

    L_work = data_length - reserved_tail
    if L_work < W:
        return [], [], {**empty_meta, "reserved_tail": reserved_tail, "reason": "L_work < W"}

    n_windows = (L_work - W) // stride + 1
    max_start = L_work - W - (n_windows - 1) * stride
    if max_start < 0:
        return [], [], {**empty_meta, "reserved_tail": reserved_tail, "reason": "max_start < 0"}

    d_min = int(np.ceil(W / 10.0))
    d_max = int(np.floor(W / 2.0))
    if d_min > d_max:
        raise ValueError(
            f"window_size={W} 过小：无法满足起点间隔 ∈ [W/10, W/2]（得到 d_min={d_min}, d_max={d_max}）"
        )

    start_ori = None
    start_aug = None
    gap = None

    # 原定 g ∈ [d_min, d_max] 需满足 g <= max_start。L_work 短时 max_start < d_min 会导致随机永远无法成功。
    if max_start == 0:
        start_ori = 0
        start_aug = 0
        gap = 0
    else:
        d_lo, d_hi = d_min, d_max
        if max_start < d_min:
            d_lo = 1
            d_hi = min(d_max, max_start)
        if d_lo > d_hi:
            start_ori = 0
            start_aug = 0
            gap = 0
        else:
            for _ in range(500):
                g = int(rng.integers(d_lo, d_hi + 1))
                upward = rng.random() < 0.5
                if upward:
                    lo, hi = 0, max_start - g
                    if lo > hi:
                        continue
                    start_ori = int(rng.integers(lo, hi + 1))
                    start_aug = start_ori + g
                else:
                    lo, hi = g, max_start
                    if lo > hi:
                        continue
                    start_ori = int(rng.integers(lo, hi + 1))
                    start_aug = start_ori - g
                gap = g
                break

    if start_ori is None:
        raise RuntimeError(
            "无法在约束下采样到合法的 (start_ori, start_aug)；请检查 window_size、overlap_ratio 与数据长度。"+L_work
        )

    res_ori = []
    res_aug = []
    for k in range(n_windows):
        i_ori = start_ori + k * stride
        i_aug = start_aug + k * stride
        res_ori.append(df.iloc[i_ori : i_ori + W, :])
        res_aug.append(df.iloc[i_aug : i_aug + W, :])

    meta = {
        "start_ori": start_ori,
        "start_aug": start_aug,
        "gap": gap,
        "n_windows": n_windows,
        "L_work": L_work,
        "stride": stride,
    }

    if enable_qc and n_windows > 0:
        arr_ori = np.array([w.values.T for w in res_ori])
        arr_aug = np.array([w.values.T for w in res_aug])
        qc_ori = window_qc(
            arr_ori,
            sfreq,
            n_channels=df.shape[1],
            bad_channel_ratio_thresh=bad_channel_ratio_thresh,
            abs_amp_thresh=abs_amp_thresh,
        )
        qc_aug = window_qc(
            arr_aug,
            sfreq,
            n_channels=df.shape[1],
            bad_channel_ratio_thresh=bad_channel_ratio_thresh,
            abs_amp_thresh=abs_amp_thresh,
        )
        qc_pair = qc_ori & qc_aug
        res_ori_f = [res_ori[i] for i in range(n_windows) if qc_pair[i]]
        res_aug_f = [res_aug[i] for i in range(n_windows) if qc_pair[i]]
        meta["qc_pass_ori"] = qc_ori
        meta["qc_pass_aug"] = qc_aug
        meta["qc_pass_pair"] = qc_pair
        print(
            f"  [random_crop_pair] QC: 总窗口 {n_windows}, 两路均通过 {qc_pair.sum()}, 丢弃 {n_windows - qc_pair.sum()}"
        )
        return res_ori_f, res_aug_f, meta

    return res_ori, res_aug, meta


def eeg_data(
    eeg_path,
    original_freq,
    target_freq,
    window_size,
    overlap_ratio,
    resampling_kind='linear',
    use_augmentation=False,
    random_seed=42,
    enable_windowing=True,
    enable_qc=True,
):
    """
    EEG数据预处理主函数

    参数:
        eeg_path: EEG文件路径
        original_freq: 原始采样率
        target_freq: 目标采样率
        window_size: 窗口大小
        overlap_ratio: 重叠比例
        resampling_kind: 插值方法
        use_augmentation: 是否启用 Random Cropping 成对增强（``segment_random_crop_pair``）
        random_seed: 仅在 ``use_augmentation=True`` 且划窗开启时用于增强路径的随机种子
        enable_windowing: 是否划窗；False 时整段序列作为一个样本
        enable_qc: 是否做窗口质量检测；False 时保留全部窗口

    返回:
        - ``use_augmentation=False``: 与 ``segment`` 相同，为 ``list[DataFrame]``
        - ``use_augmentation=True``: ``(res_ori, res_aug, meta)``，两列表长度一致且窗口一一对齐
    """
    raw = mne.io.read_raw_eeglab(eeg_path, preload=False)
    signals = raw.get_data()

    trial = []
    for i in range(signals.shape[0]):
        data = resampling(signals[i], original_freq, target_freq, kind=resampling_kind)
        trial.append(data)

    df = pd.DataFrame(trial).T

    if not enable_windowing:
        frames = [df]
        if enable_qc and df.shape[0] > 0:
            windows_array = np.array([df.values.T])
            qc_pass = window_qc(
                windows_array,
                target_freq,
                n_channels=df.shape[1],
            )
            kept = bool(qc_pass[0])
            print(f"  质量控制(整段): {'通过' if kept else '丢弃'}")
            frames = [df] if kept else []
        elif enable_qc:
            frames = []
        if use_augmentation:
            return frames, [frame.copy() for frame in frames], {
                "n_windows": len(frames),
                "windowing": False,
            }
        return frames

    if use_augmentation:
        return segment_random_crop_pair(
            df,
            window_size,
            overlap_ratio,
            sfreq=target_freq,
            enable_qc=enable_qc,
            random_seed=random_seed,
        )

    return segment(
        df, window_size, overlap_ratio, sfreq=target_freq, enable_qc=enable_qc
    )

print("预处理函数定义完成！")

预处理函数定义完成！


In [14]:
# ============================================================================
# 特征提取和保存
# ============================================================================

sub_id = 1
total_samples = 0

for sub in sorted(os.listdir(derivatives_root)):
    if 'sub-' in sub:
        sub_path = os.path.join(derivatives_root, sub, 'eeg/')

        print(f"\n处理 {sub}...")

        if USE_AUGMENTATION:
            li_ori, li_aug = [], []
            for file in sorted(os.listdir(sub_path)):
                if '.set' in file:
                    file_path = os.path.join(sub_path, file)
                    print(f"  处理文件: {file}")
                    res_ori, res_aug, _meta = eeg_data(
                        eeg_path=file_path,
                        original_freq=ORIGINAL_SAMPLING_RATE,
                        target_freq=TARGET_SAMPLING_RATE,
                        window_size=WINDOW_SIZE,
                        overlap_ratio=OVERLAP_RATIO,
                        resampling_kind=RESAMPLING_KIND,
                        use_augmentation=True,
                        random_seed=AUGMENTATION_RANDOM_SEED,
                        enable_windowing=ENABLE_WINDOWING,
                        enable_qc=ENABLE_WINDOW_QC,
                    )
                    for df_o, df_a in zip(res_ori, res_aug):
                        li_ori.append(df_o.values)
                        li_aug.append(df_a.values)

            if len(li_ori) > 0:
                arr_ori = np.array(li_ori)
                arr_aug = np.array(li_aug)
                if arr_ori.shape != arr_aug.shape:
                    raise ValueError(
                        f"{sub}: 原始/增强窗口形状不一致 {arr_ori.shape} vs {arr_aug.shape}"
                    )
                array_sub = np.stack([arr_ori, arr_aug], axis=0)
                feature_save_path = os.path.join(
                    OUTPUT_FEATURE_DIR, f'feature_{sub_id:02d}.npy'
                )
                np.save(feature_save_path, array_sub)
                print(
                    f"  保存特征(增强): {feature_save_path}, 形状: {array_sub.shape} "
                    f"(0=原始, 1=增强, N={array_sub.shape[1]})"
                )
                total_samples += array_sub.shape[1]
                sub_id += 1
            else:
                print(f"  警告: {sub} 没有有效数据")
        else:
            li_sub = []
            for file in sorted(os.listdir(sub_path)):
                if '.set' in file:
                    file_path = os.path.join(sub_path, file)
                    print(f"  处理文件: {file}")
                    res_df = eeg_data(
                        eeg_path=file_path,
                        original_freq=ORIGINAL_SAMPLING_RATE,
                        target_freq=TARGET_SAMPLING_RATE,
                        window_size=WINDOW_SIZE,
                        overlap_ratio=OVERLAP_RATIO,
                        resampling_kind=RESAMPLING_KIND,
                        enable_windowing=ENABLE_WINDOWING,
                        enable_qc=ENABLE_WINDOW_QC,
                    )
                    for df_std in res_df:
                        li_sub.append(df_std.values)

            if len(li_sub) > 0:
                array_sub = np.array(li_sub)
                feature_save_path = os.path.join(
                    OUTPUT_FEATURE_DIR, f'feature_{sub_id:02d}.npy'
                )
                np.save(feature_save_path, array_sub)
                print(f"  保存特征: {feature_save_path}, 形状: {array_sub.shape}")
                total_samples += array_sub.shape[0]
                sub_id += 1
            else:
                print(f"  警告: {sub} 没有有效数据")

print(f"\n特征提取完成！")
print(f"总样本数: {total_samples}")
print(f"处理的参与者数: {sub_id - 1}")


处理 sub-001...
  处理文件: sub-001_task-eyesclosed_eeg.set
Reading /root/autodl-tmp/ADFTD Sub88/derivatives/sub-001/eeg/sub-001_task-eyesclosed_eeg.set
  保存特征: /root/autodl-tmp/ADMamba/dataset/ADFTD3_1.0s_0.0_NoWin_NoQC/Feature/feature_01.npy, 形状: (1, 76774, 19)

处理 sub-002...
  处理文件: sub-002_task-eyesclosed_eeg.set
Reading /root/autodl-tmp/ADFTD Sub88/derivatives/sub-002/eeg/sub-002_task-eyesclosed_eeg.set
  保存特征: /root/autodl-tmp/ADMamba/dataset/ADFTD3_1.0s_0.0_NoWin_NoQC/Feature/feature_02.npy, 形状: (1, 101516, 19)

处理 sub-003...
  处理文件: sub-003_task-eyesclosed_eeg.set
Reading /root/autodl-tmp/ADFTD Sub88/derivatives/sub-003/eeg/sub-003_task-eyesclosed_eeg.set
  保存特征: /root/autodl-tmp/ADMamba/dataset/ADFTD3_1.0s_0.0_NoWin_NoQC/Feature/feature_03.npy, 形状: (1, 39180, 19)

处理 sub-004...
  处理文件: sub-004_task-eyesclosed_eeg.set
Reading /root/autodl-tmp/ADFTD Sub88/derivatives/sub-004/eeg/sub-004_task-eyesclosed_eeg.set
  保存特征: /root/autodl-tmp/ADMamba/dataset/ADFTD3_1.0s_0.0_NoWin_NoQC/Featur

In [1]:
# ============================================================================
# 验证保存的数据（可选）
# ============================================================================
def count_original_samples(shape):
    """增强 layout (2, N, T, C) 时只统计第 1 维原始样本数 N"""
    if len(shape) == 4 and shape[0] == 2:
        return shape[1]
    return shape[0]

# 验证标签
label_path = os.path.join(OUTPUT_LABEL_DIR, 'label.npy')
if os.path.exists(label_path):
    loaded_labels = np.load(label_path)
    print("标签验证:")
    print(f"  标签形状: {loaded_labels.shape}")
    print(f"  标签预览:\n{loaded_labels}")

# 验证特征
print("\n特征验证:")
all_samples = 0
feature_files = sorted([f for f in os.listdir(OUTPUT_FEATURE_DIR) if f.endswith('.npy')])
for file in feature_files:
    file_path = os.path.join(OUTPUT_FEATURE_DIR, file)
    shape = np.load(file_path).shape
    n_samples = count_original_samples(shape)
    print(f"  {file}: {shape} -> 原始样本数: {n_samples}")
    all_samples += n_samples

print(f"\n原始样本总数: {all_samples}")
if os.path.exists(label_path):
    print(f"label 条目数: {len(loaded_labels)}")

NameError: name 'os' is not defined